# 04 — Lab's from-scratch Transformer

Reproduces the transformer architecture from the Lab 4 brief (a Keras `TransformerBlock` + token/positional embedding, no pre-training) and trains it on our cleaned dataset.

This serves as a **fourth baseline**: it shows what the lab's own approach achieves on the same splits, so that the BERT-family fine-tuning gains in notebooks 05–06 are anchored against a fair reference point.

Lecturer reported 95.7% accuracy on the raw lab dataset. Note that our cleaned dataset has hashtags removed (which were the original source of the stance label), so the task here is genuinely harder.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

from src.data import load_splits
from src.eval import evaluate, append_metrics, pretty_report, LABELS

tf.random.set_seed(42)
np.random.seed(42)

FIG_DIR = PROJECT_ROOT / 'results' / 'figures' / 'lab_transformer'
FIG_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR = PROJECT_ROOT / 'results' / 'predictions'
MODEL_DIR = PROJECT_ROOT / 'results' / 'models'

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110
print(f'tensorflow: {tf.__version__}')

## 1. Lab's TransformerBlock and TokenAndPositionEmbedding (verbatim from Lab 4 brief)

In [ ]:
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1, **kw):
        super().__init__(**kw)
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation='relu'),
            layers.Dense(embed_dim),
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def call(self, inputs, training=None):
        attn = self.att(inputs, inputs, inputs)
        attn = self.dropout1(attn, training=training)
        out1 = self.layernorm1(inputs + attn)
        ffn_out = self.ffn(out1)
        ffn_out = self.dropout2(ffn_out, training=training)
        return self.layernorm2(out1 + ffn_out)


class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim, **kw):
        super().__init__(**kw)
        self.token_emb = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb = layers.Embedding(input_dim=maxlen, output_dim=embed_dim)

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        return self.token_emb(x) + self.pos_emb(positions)

## 2. Load splits, label-encode, vectorise

In [ ]:
VOCAB_SIZE = 30_000
MAX_LEN = 96

vectorizer = layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode='int',
    output_sequence_length=MAX_LEN,
)
vectorizer.adapt(tf.constant(X_train_text, dtype=tf.string))
vocab = vectorizer.get_vocabulary()
actual_vocab = len(vocab)
print(f'vocab size: {actual_vocab:,}, max_len: {MAX_LEN}')


## 3. Build the lab transformer

Hyperparameters from the Lab 4 brief: 128-dim embeddings, 2 attention heads, 512-unit feed-forward, dropout 0.1.

In [ ]:
EMBED_DIM = 128
NUM_HEADS = 2
FF_DIM = 512
DROPOUT = 0.1

def build_lab_transformer():
    inputs = layers.Input(shape=(1,), dtype=tf.string)
    x = vectorizer(inputs)
    x = TokenAndPositionEmbedding(MAX_LEN, actual_vocab, EMBED_DIM)(x)
    x = TransformerBlock(EMBED_DIM, NUM_HEADS, FF_DIM, rate=DROPOUT)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(DROPOUT)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(DROPOUT)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    return models.Model(inputs, outputs, name='lab_transformer')

model = build_lab_transformer()
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)
model.summary()

## 4. Train

In [ ]:
EPOCHS = 4
BATCH = 256

X_train_t = tf.constant(X_train_text, dtype=tf.string)
X_val_t   = tf.constant(X_val_text,   dtype=tf.string)

early = callbacks.EarlyStopping(monitor='val_accuracy', patience=2, restore_best_weights=True)

t0 = time.time()
history = model.fit(
    X_train_t, y_train,
    validation_data=(X_val_t, y_val),
    epochs=EPOCHS,
    batch_size=BATCH,
    callbacks=[early],
    verbose=2,
)
print(f'train time: {time.time() - t0:.1f}s')

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(history.history['loss'], label='train')
axes[0].plot(history.history['val_loss'], label='val')
axes[0].set_title('Loss'); axes[0].set_xlabel('epoch'); axes[0].legend()
axes[1].plot(history.history['accuracy'], label='train')
axes[1].plot(history.history['val_accuracy'], label='val')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('epoch'); axes[1].legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'training_curves.png', bbox_inches='tight')
plt.show()


## 5. Evaluate on test

In [ ]:
X_test_t = tf.constant(X_test_text, dtype=tf.string)
y_proba_test = model.predict(X_test_t, batch_size=512, verbose=0).ravel()
y_pred_int = (y_proba_test >= 0.5).astype(int)
y_pred_str = np.array([int_to_label[i] for i in y_pred_int])
y_test_str = np.array([int_to_label[i] for i in y_test])

metrics, cm = evaluate('lab_transformer', y_test_str, y_pred_str, y_proba=y_proba_test)
print(pretty_report(y_test_str, y_pred_str))
append_metrics(metrics)

pd.DataFrame({
    'y_true': y_test_str,
    'y_pred': y_pred_str,
    'proba_sceptic': y_proba_test,
}).to_parquet(PRED_DIR / 'lab_transformer.parquet', index=False)

fig, ax = plt.subplots(figsize=(4.2, 3.8))
sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues',
            xticklabels=LABELS, yticklabels=LABELS, cbar=False, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Lab transformer — confusion (test)')
plt.tight_layout()
plt.savefig(FIG_DIR / 'cm_lab_transformer.png', bbox_inches='tight')
plt.show()

model.save(MODEL_DIR / 'lab_transformer.keras')


In [ ]:
results = pd.read_csv(PROJECT_ROOT / 'results' / 'metrics.csv')
results.round(4)